In [1]:
import pandas as pd
import numpy as np
import re

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import RobustScaler

In [2]:
train_path = '/kaggle/input/mars-environmental-dynamics-analyzer-meda-virtual-sensor-recovery/train.parquet'
test_path = '/kaggle/input/mars-environmental-dynamics-analyzer-meda-virtual-sensor-recovery/test.parquet'

train = pd.read_parquet(train_path)
test = pd.read_parquet(test_path)

initial_cols = train.columns
train = train.dropna(axis=1, how='all')


In [3]:
def lmst_to_decimal_sol(lmst_str):
    if pd.isna(lmst_str):
        return None
    
    match = re.match(r'(\d+)M(\d+):(\d+):([\d.]+)', str(lmst_str))
    if match:
        sol = int(match.group(1))
        hours = int(match.group(2))
        minutes = int(match.group(3))
        seconds = float(match.group(4))
        
        decimal_part = (hours / 24) + (minutes / (24 * 60)) + (seconds / (24 * 3600))
        return sol + decimal_part
    return None


def create_time_features(df):
    df['sol_decimal'] = df['LMST'].apply(lmst_to_decimal_sol)
    
    df['day_fraction'] = df['sol_decimal'] % 1
    
    df['sin_time'] = np.sin(2 * np.pi * df['day_fraction'])
    df['cos_time'] = np.cos(2 * np.pi * df['day_fraction'])
    
    df['sol_int'] = df['sol_decimal'].astype(int)
    
    return df

train = create_time_features(train)
test = create_time_features(test)

train = train[(train['PRESSURE'] > 600) & (train['PRESSURE'] < 900)].copy()

In [4]:
train['time_bucket'] = (train['day_fraction'] * 100).round()
test['time_bucket'] = (test['day_fraction'] * 100).round()

pressure_map = train.groupby('time_bucket')['PRESSURE'].mean()

train['avg_pressure_time'] = train['time_bucket'].map(pressure_map)
test['avg_pressure_time'] = test['time_bucket'].map(pressure_map)

For pressure map: https://www.kaggle.com/code/irendeika/pressure-breath-atmospheric-tides

In [5]:
features = train.select_dtypes(include=[np.number]).columns.drop(['PRESSURE'], errors='ignore').tolist()

train_filled = train[features].fillna(train[features].median())
test_filled = test[features].fillna(train[features].median())

scaler = RobustScaler()
X_train = scaler.fit_transform(train_filled)
X_test = scaler.transform(test_filled)
y_train = train['PRESSURE'].values.reshape(-1, 1)

y_scaler = RobustScaler()
y_train_scaled = y_scaler.fit_transform(train['PRESSURE'].values.reshape(-1, 1))

In [6]:
class MarsPressureDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

class PressurePredictor(nn.Module):
    def __init__(self, input_dim):
        super(PressurePredictor, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1) 
        )
        
    def forward(self, x):
        return self.net(x)

model = PressurePredictor(len(features))
dataset = MarsPressureDataset(X_train, y_train_scaled)
dataloader = DataLoader(dataset, batch_size=1024, shuffle=True)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [7]:
model.train()
epochs = 5 

for epoch in range(epochs):
    total_loss = 0
    for batch_x, batch_y in dataloader:
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, MSE Loss: {total_loss/len(dataloader):.4f}")

Epoch 1/5, MSE Loss: 0.0073
Epoch 2/5, MSE Loss: 0.0031
Epoch 3/5, MSE Loss: 0.0025
Epoch 4/5, MSE Loss: 0.0022
Epoch 5/5, MSE Loss: 0.0020


In [8]:
model.eval()
test_dataset = MarsPressureDataset(X_test)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

predictions = []
with torch.no_grad():
    for batch_x in test_loader:
        preds = model(batch_x)
        predictions.extend(preds.numpy().flatten())

real_predictions = y_scaler.inverse_transform(np.array(predictions).reshape(-1, 1))

submission = pd.DataFrame({
    'row_id': test['SCLK'],
    'PRESSURE': real_predictions.flatten()
})

submission.to_csv('DL_Pressure_map_SAMPLE.csv', index=False)

In [9]:
submission

,row_id,PRESSURE
0,684737856,811.593872
1,684737857,811.623718
2,684737858,811.634155
3,684737859,811.576477
4,684737860,811.669128
...,...,...
3949985,693611920,825.128052
3949986,693611921,825.079590
3949987,693611922,824.927429
3949988,693611923,824.871338
